In [1]:
# =========================================
#  Detectable Computation via Syndromes
#  (Programs injected between stabilize() rounds)
# =========================================
from __future__ import annotations
import json, csv, time, math, itertools
from collections import defaultdict, Counter
from pathlib import Path
import numpy as np

from qiskit import qasm3
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler

In [2]:
# ---------- USER CONFIG (EDIT THESE) ----------
CONFIG_PATH = Path("../../config.json")   # your IBM Cloud token/instance file
KEY = 'devon' # can also equal 'lab
OUTDIR = Path("./out_program_leakage") # where artifacts are saved
OUTDIR.mkdir(parents=True, exist_ok=True)

BACKENDS = [
    "ibm_brisbane",
    # add others if you like
]

CODES = ["surface"]       # can add "steane","shor" later
D = 5                     # surface code distance
R = 3                     # rounds of stabilize()
SHOTS = 1000
OPT_LEVEL = 2
LAYOUT_METHODS = ["sabre"]
SEED_TRANSPILE = 20251022  # deterministic transpiler for fair A/B comparisons

# Programs per initial state (we fix logical state for this pivot)
RUNS_PER_PROGRAM = 2
PROGRAM_SET = ["idle", "X", "Z"]   # start with two; you can add more below
FIXED_STATE = {"theta": 0.0, "phi": 0.0}  # fixed logical prep


In [3]:
# ---------- PROGRAM HOOKS (operate on XXZZQubit) ----------
def prog_idle(q):   # do nothing
    return

def prog_logical_X(q):
    # logical X of qtcodes.XXZZQubit
    q.x()

def prog_logical_Z(q):
    q.z()

PROGRAMS = {
    "idle": prog_idle,
    "X":    prog_logical_X,
    "Z":    prog_logical_Z,
}

# ---------- HELPERS ----------
def sanitize_angle(angle: float) -> str:
    eps = 1e-6
    if abs(angle) < eps: return "0"
    if abs(angle - math.pi/2) < eps: return "pi2"
    if abs(angle - math.pi) < eps: return "pi"
    if abs(angle - 3*math.pi/2) < eps: return "3pi2"
    return str(round(angle, 3)).replace(".", "_").replace("-", "m")

def total_variation_distance(p_counts: Counter, q_counts: Counter) -> float:
    keys = set(p_counts) | set(q_counts)
    p_total = sum(p_counts.values()); q_total = sum(q_counts.values())
    tv = 0.0
    for k in keys:
        pv = p_counts.get(k, 0) / p_total if p_total else 0.0
        qv = q_counts.get(k, 0) / q_total if q_total else 0.0
        tv += abs(pv - qv)
    return 0.5 * tv

def _perm_from_ids(ids):
    """Return a permutation (list of indices) that sorts ids stably."""
    return [i for i, _ in sorted(enumerate(ids), key=lambda t: t[1])]

# ---- canonicalization helpers (if stabilizer IDs available) ----
def _canonicalize_round(bits: str, meta: dict) -> str:
    # Split the round's bitstring into Z- and X-blocks
    if meta["order"] == "ZX":
        z, x = bits[:meta["num_z"]], bits[meta["num_z"]:]
    else:
        x, z = bits[:meta["num_x"]], bits[meta["num_x"]:meta["num_x"] + meta["num_z"]]

    # Build permutations using pure-Python sorting on the IDs
    if "z_ids" in meta and len(meta["z_ids"]) == len(z):
        z_perm = _perm_from_ids(meta["z_ids"])
    else:
        z_perm = list(range(len(z)))

    if "x_ids" in meta and len(meta["x_ids"]) == len(x):
        x_perm = _perm_from_ids(meta["x_ids"])
    else:
        x_perm = list(range(len(x)))

    # Apply permutations
    z_can = "".join(z[i] for i in z_perm)
    x_can = "".join(x[i] for i in x_perm)
    return z_can + x_can


def extract_split_syndromes_canonical(reg2strings: dict[str, list[str]],
                                      syn_reg_names: list[str],
                                      syn_meta: list[dict]) -> tuple[list[str]|None, list[str]|None]:
    """Return (z_joined, x_joined) canonicalized per round. If meta lacks IDs, return (None,None)."""
    have_ids = all(("num_z" in m and "num_x" in m and "order" in m and
                    "z_ids" in m and "x_ids" in m and
                    len(m["z_ids"]) == m["num_z"] and len(m["x_ids"]) == m["num_x"])
                   for m in syn_meta)
    if not have_ids:
        print('NO IDS')
        return None, None

    z_rounds, x_rounds = [], []
    for rn, meta in zip(syn_reg_names, syn_meta):
        raw = reg2strings[rn]  # list[str] for this round
        canon = [_canonicalize_round(s, meta) for s in raw]
        if meta["order"] == "ZX":
            num_z = meta["num_z"]
            z_bits = [s[:num_z] for s in canon]
            x_bits = [s[num_z:]  for s in canon]
        else:
            num_x = meta["num_x"]
            x_bits = [s[:num_x] for s in canon]
            z_bits = [s[num_x:]  for s in canon]
        z_rounds.append(z_bits); x_rounds.append(x_bits)

    shots = len(z_rounds[0]) if z_rounds else 0
    z_joined = ["".join(z_rounds[t][s] for t in range(len(z_rounds))) for s in range(shots)]
    x_joined = ["".join(x_rounds[t][s] for t in range(len(x_rounds))) for s in range(shots)]
    return z_joined, x_joined

# ---- permutation-invariant features: (wZ, wX) ----
def weight_histograms(reg2strings: dict[str, list[str]],
                      syn_reg_names: list[str],
                      syn_meta: list[dict]) -> Counter:
    """Counter over (total_wZ, total_wX) across all rounds for a circuit’s shots."""
    z_rounds, x_rounds = [], []
    for rn, meta in zip(syn_reg_names, syn_meta):
        full = reg2strings[rn]
        if meta["order"] == "ZX":
            num_z = meta["num_z"]
            z_bits = [s[:num_z] for s in full]
            x_bits = [s[num_z:]  for s in full]
        else:
            num_x = meta["num_x"]
            x_bits = [s[:num_x] for s in full]
            z_bits = [s[num_x:]  for s in full]
        z_rounds.append(z_bits); x_rounds.append(x_bits)

    shots = len(z_rounds[0]) if z_rounds else 0
    hist = Counter()
    for s in range(shots):
        wZ = sum(zi[s].count("1") for zi in z_rounds)
        wX = sum(xi[s].count("1") for xi in x_rounds)
        hist[(wZ, wX)] += 1
    return hist




In [4]:
# ---------- BUILDERS ----------
def _ids_from_geometry(lattice):
    geo = lattice.geometry  # {'mx': [[anc,n1,n2,n3,n4], ...], 'mz': [...]}

    def _rows_to_ids(rows, tag):
        ids = []
        for row in rows:
            anc = int(row[0])
            nbrs = tuple(-1 if v is None else int(v) for v in row[1:])  # replace None for sortability
            ids.append((tag, anc, *nbrs))  # FLATTENED: ('Z', 3, 10, 11, 15, 16)
        return ids

    x_ids = _rows_to_ids(geo['mx'], 'X')
    z_ids = _rows_to_ids(geo['mz'], 'Z')
    return z_ids, x_ids


def build_surface_circuit(theta: float, phi: float, d: int = D, rounds: int = R, program_id: str = "idle"):
    """
    Surface code via qtcodes.XXZZQubit with program hook after each stabilize().
    Returns: (qc, syn_reg_names, syn_meta)
    """
    from qtcodes import XXZZQubit

    name = f"surface_d{d}_theta{sanitize_angle(theta)}_phi{sanitize_angle(phi)}_{program_id}"
    q = XXZZQubit({'d': d}, name=name)

    # Prepare the single logical qubit state on its input physical qubit
    q.circ.u(theta, phi, 0, q.circ.qubits[0])

    syn_reg_names, syn_meta = [], []
    program_fn = PROGRAMS[program_id]

    for _t in range(rounds):
        # 1) Stabilize (measure syndromes)
        q.stabilize()

        # 2) Record meta for canonicalization (now using geometry-based IDs)
        T = q.lattice.params["T"]
        creg = q.lattice.cregisters[f"syndrome{T}"]
        num_syn = q.lattice.params["num_syn"]
        num_z = int(num_syn[q.lattice.SYNZ]); num_x = int(num_syn[q.lattice.SYNX])

        z_ids, x_ids = _ids_from_geometry(q.lattice)    # <<-- NEW: stable IDs from geometry
        assert len(z_ids) == num_z and len(x_ids) == num_x, "ID count mismatch vs num_syn"

        syn_reg_names.append(creg.name)
        syn_meta.append({
            "order": "ZX",
            "num_z": num_z, "num_x": num_x,
            "z_ids": z_ids, "x_ids": x_ids,
        })

        # 3) Computation/program between rounds
        program_fn(q)

    return q.circ, syn_reg_names, syn_meta

CODE_BUILDERS = {
    "surface": build_surface_circuit,
    # "steane": build_steane_circuit,  # add later
    # "shor":   build_shor_circuit,    # add later
}

In [5]:
# ---------- PARAM GRID (programs × runs) ----------
def make_params_list() -> list[dict]:
    params = []
    for prog in PROGRAM_SET:
        for r in range(RUNS_PER_PROGRAM):
            params.append({
                "theta": FIXED_STATE["theta"],
                "phi":   FIXED_STATE["phi"],
                "program_id": prog,
                "run_id": r,
            })
    return params

def build_and_pack_circuits(code_name: str, params_list: list[dict]):
    builder = CODE_BUILDERS[code_name]
    metadata, circuit_objs = [], []
    for pp in params_list:
        theta, phi = float(pp["theta"]), float(pp["phi"])
        run_id = pp.get("run_id", 0)
        program_id = pp.get("program_id", "idle")
        qc, syn_regs, syn_meta = builder(theta, phi, rounds=R, program_id=program_id)
        metadata.append({
            "code": code_name,
            "theta": theta,
            "phi": phi,
            "program_id": program_id,
            "run_id": run_id,
            "syn_reg_names": syn_regs,
            "syn_meta": syn_meta,
            "qasm3": qasm3.dumps(qc),
        })
        circuit_objs.append(qc)
    return metadata, circuit_objs


In [6]:
# ---------- TV assembly for program keys ----------
def _unpack_prog_key(k):
    # keys are (code, program_id, run_id)
    return k[0], k[1], k[2]

def compute_tv_rows_program(tag: str, per_counts: dict, backend_name: str):
    """Pairwise TVs for map[(code, program, run) -> Counter]."""
    rows = []
    for code in CODES:
        keys = [k for k in per_counts if k[0] == code]
        for (k1, k2) in itertools.combinations(keys, 2):
            (c1, p1, r1), (c2, p2, r2) = k1, k2
            tv = total_variation_distance(per_counts[k1], per_counts[k2])
            rows.append({
                "backend": backend_name,
                "code": c1,
                "program1": p1, "run1": r1,
                "program2": p2, "run2": r2,
                "same_program": (p1 == p2),
                "TV": tv,
                "view": tag,
            })
    return rows

def merge_program_rows(backend_name: str, rows_str_all, rows_w_all, outpath: Path):
    """Append merged program TV rows (string + weight)."""
    def keyify(r):
        return (r["code"], r["program1"], r["run1"], r["program2"], r["run2"])
    m_str = {keyify(r): r["TV"] for r in rows_str_all}
    m_w   = {keyify(r): r["TV"] for r in rows_w_all}
    allk  = set(m_str) | set(m_w)
    merged = []
    for k in sorted(allk):
        code, p1, r1, p2, r2 = k
        merged.append({
            "backend": backend_name,
            "code": code,
            "program1": p1, "run1": r1,
            "program2": p2, "run2": r2,
            "same_program": (p1 == p2),
            "TV_all_str": m_str.get(k, ""),   # blank if canonical strings not available
            "TV_w_all":   m_w.get(k, ""),
        })
    file_exists = outpath.exists()
    with open(outpath, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=[
            "backend","code","program1","run1","program2","run2",
            "same_program","TV_all_str","TV_w_all"
        ])
        if not file_exists: w.writeheader()
        w.writerows(merged)
    print(f"Appended {len(merged)} rows -> {outpath.name}")


In [7]:
# =========================================
#                  MAIN
# =========================================
def main():
    # Auth
    with open(CONFIG_PATH) as f:
        cfg = json.load(f)[KEY]
    service = QiskitRuntimeService(
        channel="ibm_cloud",
        token=cfg["api_key"],
        instance=cfg["ibm_instance_crn"],
    )

    params = make_params_list()

    for backend_name in BACKENDS:
        backend = service.backend(backend_name)
        print(f"\n== Backend: {backend_name} ==")

        # ---------- build all circuits ----------
        all_metadata, all_circuits = [], []
        for code in CODES:
            md, circs = build_and_pack_circuits(code, params)
            all_metadata.extend(md); all_circuits.extend(circs)

        # ---------- transpile (deterministic) ----------
        transpiled, saved_md = [], []
        rng = np.random.default_rng(12345)

        for meta, qc in zip(all_metadata, all_circuits):
            code = meta["code"]
            prog = meta["program_id"]
            theta = meta["theta"]
            phi = meta["phi"]
            run_id = meta["run_id"]

            # create a unique path for this program/state combo
            base_tag = f"{backend_name}_{code}_theta{theta:.3f}_phi{phi:.3f}_{prog}"
            pkl_path = OUTDIR / f"{base_tag}_transpiled.pkl"

            if run_id == 0:
                # --- first run: transpile and save ---
                initial_layout = list(rng.permutation(qc.num_qubits))
                pm = generate_preset_pass_manager(
                    optimization_level=OPT_LEVEL,
                    backend=backend,
                    layout_method=LAYOUT_METHODS[0],
                    initial_layout=initial_layout,
                    seed_transpiler=SEED_TRANSPILE,
                )
                tqc = pm.run(qc)

                # save to disk for reuse
                with open(pkl_path, "wb") as f:
                    import pickle as p
                    p.dump({"tqc": tqc, "initial_layout": initial_layout}, f)

                print(f"Saved transpiled circuit for {prog} to {pkl_path.name}")

            else:
                # --- later runs: load cached transpiled circuit ---
                if pkl_path.exists():
                    import pickle as p
                    with open(pkl_path, "rb") as f:
                        data = p.load(f)
                    tqc = data["tqc"]
                    initial_layout = data["initial_layout"]
                    print(f"Loaded cached transpiled circuit for {prog} from {pkl_path.name}")
                else:
                    raise FileNotFoundError(
                        f"No cached transpiled circuit found for {base_tag} (expected {pkl_path})"
                    )

            transpiled.append(tqc)
            m = dict(meta)
            m["backend"] = backend_name
            m["layout_method"] = LAYOUT_METHODS[0]
            m["initial_layout"] = initial_layout
            saved_md.append(m)

        # Save circuits bundle
        pkl_path = OUTDIR / f"{backend_name}_circuits_programs.pkl"
        with open(pkl_path, "wb") as f:
            import pickle as p
            p.dump((saved_md, transpiled), f)
        print(f"==\tSaved ciccuit to: {pkl_path} ==")

        # ---------- run ----------
        sampler = Sampler(mode=backend)
        print(f"\tSubmitting {len(transpiled)} circuits to {backend_name} with {SHOTS} shots each …")
        job = sampler.run(transpiled, shots=SHOTS)
        submit_ts = time.time()
        results = job.result()
        print("\t\tJob complete:", job.job_id())

        # ---------- aggregate per (code, program, run) ----------
        per_prog_counts_all_str = defaultdict(Counter)  # canonicalized strings (Z+X)
        per_prog_counts_w_all   = defaultdict(Counter)  # (wZ,wX) histogram

        per_shot_rows = []  # optional detailed CSV

        for idx, res in enumerate(results):
            meta = saved_md[idx]
            key = (meta["code"], meta["program_id"], meta["run_id"])
            syn_regs = meta["syn_reg_names"]
            syn_meta = meta["syn_meta"]

            # map reg->list[str]
            reg2strings = {reg: vals.get_bitstrings() for reg, vals in res.data.items()}

            # canonical strings (if IDs present)
            z_joined, x_joined = extract_split_syndromes_canonical(reg2strings, syn_regs, syn_meta)
            if z_joined is not None:
                combined = [z + x for z, x in zip(z_joined, x_joined)]
                per_prog_counts_all_str[key].update(combined)

                # optional per-shot logging
                for sZ, sX in zip(z_joined, x_joined):
                    per_shot_rows.append({
                        "backend": meta["backend"], "code": meta["code"],
                        "program": meta["program_id"], "run": meta["run_id"],
                        "syndrome_Z": sZ, "syndrome_X": sX
                    })

            # permutation-invariant weights (always)
            wh = weight_histograms(reg2strings, syn_regs, syn_meta)
            per_prog_counts_w_all[key].update(wh)

        # ---------- save raw counts ----------
        raw_json = {
            "backend": backend_name,
            "shots": SHOTS,
            "rounds": R,
            "timestamp": submit_ts,
            "counts_all_str": { f"{c}|prog={p}|run={r}": dict(cnt)
                                for (c,p,r), cnt in per_prog_counts_all_str.items() },
            "counts_w_all":   { f"{c}|prog={p}|run={r}": { str(k): v for k,v in cnt.items() }
                                for (c,p,r), cnt in per_prog_counts_w_all.items() },
        }
        with open(OUTDIR / f"{backend_name}_syndrome_counts_program.json", "w") as f:
            json.dump(raw_json, f, indent=2)

        if per_shot_rows:
            with open(OUTDIR / f"{backend_name}_per_shot_program.csv", "w", newline="") as f:
                w = csv.DictWriter(f, fieldnames=["backend","code","program","run","syndrome_Z","syndrome_X"])
                w.writeheader(); w.writerows(per_shot_rows)

        # ---------- compute TVs across programs (signal) and across runs of same program (baseline) ----------
        rows_str_all = compute_tv_rows_program("str_all", per_prog_counts_all_str, backend_name)
        rows_w_all   = compute_tv_rows_program("w_all",   per_prog_counts_w_all,   backend_name)

        # ---------- merged program-level TV CSV (append) ----------
        merged_path = OUTDIR / "tv_program_merged.csv"
        merge_program_rows(backend_name, rows_str_all, rows_w_all, merged_path)

        print(f"Saved: {pkl_path.name}, {backend_name}_syndrome_counts_program.json, "
              f"{'per_shot_program.csv, ' if per_shot_rows else ''}{merged_path.name}")

if __name__ == "__main__":
    main()


== Backend: ibm_brisbane ==
Saved transpiled circuit for idle to ibm_brisbane_surface_theta0.000_phi0.000_idle_transpiled.pkl
Loaded cached transpiled circuit for idle from ibm_brisbane_surface_theta0.000_phi0.000_idle_transpiled.pkl
Saved transpiled circuit for X to ibm_brisbane_surface_theta0.000_phi0.000_X_transpiled.pkl
Loaded cached transpiled circuit for X from ibm_brisbane_surface_theta0.000_phi0.000_X_transpiled.pkl
Saved transpiled circuit for Z to ibm_brisbane_surface_theta0.000_phi0.000_Z_transpiled.pkl
Loaded cached transpiled circuit for Z from ibm_brisbane_surface_theta0.000_phi0.000_Z_transpiled.pkl
==	Saved ciccuit to: out_program_leakage/ibm_brisbane_circuits_programs.pkl ==
	Submitting 6 circuits to ibm_brisbane with 1000 shots each …
		Job complete: d3t6db1sg33c73de286g
Appended 15 rows -> tv_program_merged.csv
Saved: ibm_brisbane_circuits_programs.pkl, ibm_brisbane_syndrome_counts_program.json, per_shot_program.csv, tv_program_merged.csv
